In [ ]:
import numpy as np

# import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from students.ushatov.lesson2 import Exercise

In [ ]:
lr_list = [round(0.01 + i * 0.01, 2) for i in range(500)]
print(lr_list)

batch_list = [round(1 + i * 1, 2) for i in range(90)]
print(batch_list)

epochs = 25

In [ ]:
X, y = load_iris(return_X_y=True)

#  y = 0 (Setosa),  y = 1 (Versicolor), y = 2 (Virginica)
y_binary = (y == 0).astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_binary, test_size=0.4, random_state=32, stratify=y_binary, shuffle=True
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=32, stratify=y_temp, shuffle=True
)

In [ ]:
# Нормализация
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_val_norm = scaler.transform(X_val)
X_test_norm = scaler.transform(X_test)

In [ ]:
def train(lr, batch_size, Rng, max_epochs=25):
    model = Exercise.create_logistic_model(num_features=4, rng=Rng)

    for epoch in range(1, max_epochs + 1):
        Exercise.fit(model, X_train_norm, y_train, lr, 1, batch_size)

        auroc = model.metric(X_val_norm, y_val, "AUROC")

        if auroc == 1.0:
            auroc = model.metric(X_test_norm, y_test, "AUROC")
            print(f"Accuracy 1 достигнута на эпохе {epoch} lr:{lr} batch:{batch_size} AUROC:{auroc}")
            return {"lr": lr, "batch": batch_size, "epoch": epoch, "auroc": auroc}

    auroc = model.metric(X_val_norm, y_val, "AUROC")
    print(f"Accuracy 1 не достигнута на lr:{lr} batch:{batch_size}")
    return {"lr": lr, "batch": batch_size, "epoch": epoch, "auroc": auroc}

In [ ]:
total = len(lr_list) * len(batch_list)
data = [None] * total
len_batch = len(batch_list)
best_comb = None
rng = np.random.default_rng(2)

for i, lr in enumerate(lr_list):
    for b, batch in enumerate(batch_list):
        result = train(lr, batch, rng, epochs)
        data[i * len_batch + b] = result
        if (
            best_comb is None
            or result["epoch"] < best_comb["epoch"]
            or (result["epoch"] == best_comb["epoch"] and result["auroc"] > best_comb["auroc"])
        ):
            best_comb = result

if best_comb is not None:
    print("Лучшая комбинация (минимум эпох, при равенстве — максимум AUROC):")
    print(
        f"lr={best_comb['lr']:.2f}, batch={best_comb['batch']}, "
        f"эпох до accuracy=1: {best_comb['epoch']}, "
        f"финальный AUROC: {best_comb['auroc']:.4f}"
    )
else:
    print("Ни одна комбинация не достигла accuracy=1 за указанное число эпох.")

In [ ]:
one_epoch_count = sum(1 for d in data if d is not None and d["epoch"] == 1)
percent_one_epoch = (one_epoch_count / total) * 100
print(f"Всего комбинаций: {total}")
print(f"Из них достигли auroc = 1 за 1 эпоху: {one_epoch_count} ({percent_one_epoch:.2f}%)")